# Handwritten Digit Recognition using Fine-Tuned ResNet

This notebook follows the same fine-tuning workflow pattern: load data, preprocess, load a pretrained model, configure training, fine-tune, evaluate, save, reload, and predict.

**Dataset:** MNIST (digits 0–9).  
**Model:** Pretrained ResNet-18 (`microsoft/resnet-18`).  
**No external database is required.** The dataset is downloaded automatically from Hugging Face.

## 1. Install required libraries

In [ ]:
!pip install -U datasets transformers torch accelerate scikit-learn matplotlib pillow

## 2. Import libraries

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from PIL import Image

## 3. Load the MNIST dataset

In [ ]:
dataset = load_dataset("mnist")

print(dataset)
print("Training records:", len(dataset["train"]))
print("Testing records:", len(dataset["test"]))

## 4. Select training and testing samples

In [ ]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(2000))

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

## 5. Display sample images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(train_dataset[i]["image"], cmap="gray")
    ax.set_title(f"Label: {train_dataset[i]['label']}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Load the image processor

In [ ]:
model_checkpoint = "microsoft/resnet-18"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)

## 7. Preprocess the images

In [ ]:
def preprocess_images(examples):
    images = [image.convert("RGB") for image in examples["image"]]
    inputs = image_processor(images=images, return_tensors="np")
    inputs["labels"] = examples["label"]
    return inputs

train_dataset = train_dataset.map(
    preprocess_images,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_dataset = test_dataset.map(
    preprocess_images,
    batched=True,
    remove_columns=test_dataset.column_names
)

train_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

print(train_dataset[0].keys())

## 8. Load the pretrained image classification model

In [ ]:
id2label = {i: str(i) for i in range(10)}
label2id = {str(i): i for i in range(10)}

model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=10,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

## 9. Freeze and unfreeze model layers

In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model.classifier.parameters():
    parameter.requires_grad = True

for name, parameter in model.named_parameters():
    if name.startswith("resnet.encoder.stages.3"):
        parameter.requires_grad = True

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_parameters = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)

## 10. Define evaluation metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

## 11. Set training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./mnist-resnet-results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

## 12. Create the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

## 13. Fine-tune the model

In [ ]:
trainer.train()

## 14. Evaluate the model

In [ ]:
evaluation_results = trainer.evaluate()
print(evaluation_results)

## 15. Display the confusion matrix

In [ ]:
predictions_output = trainer.predict(test_dataset)
predicted_labels = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

cm = confusion_matrix(true_labels, predicted_labels)
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=list(range(10))
)
display.plot(cmap="Blues", xticks_rotation="vertical")
plt.title("MNIST Confusion Matrix")
plt.show()

## 16. Save the fine-tuned model and image processor

In [ ]:
save_directory = "./mnist-digit-resnet"

trainer.save_model(save_directory)
image_processor.save_pretrained(save_directory)

print("Model saved to:", save_directory)

## 17. Reload the saved model

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(save_directory)
model = AutoModelForImageClassification.from_pretrained(save_directory)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Model loaded on:", device)

## 18. Predict a handwritten digit image

In [ ]:
def predict_digit(image):
    if not isinstance(image, Image.Image):
        image = Image.open(image)

    image = image.convert("RGB")
    inputs = image_processor(images=image, return_tensors="pt")
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        predicted_digit = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0, predicted_digit].item()

    print("Predicted digit:", predicted_digit)
    print("Confidence:", f"{confidence * 100:.2f}%")

    plt.imshow(image.convert("L"), cmap="gray")
    plt.axis("off")
    plt.show()

    return predicted_digit

## 19. Upload an image and test the model

In [ ]:
from PIL import Image
from IPython.display import display

try:
    from google.colab import files

    uploaded = files.upload()
    image_path = next(iter(uploaded))
    predict_digit(image_path)

except ImportError:
    print("Google Colab upload is unavailable.")
    print("For local Jupyter, enter the path to an image file:")
    image_path = input("Image path: ").strip()
    predict_digit(image_path)

## Notes

- The notebook downloads MNIST automatically; no separate database is needed.
- The notebook uses 10,000 training images and 2,000 test images to keep the first run manageable. Students can increase these counts after the workflow works.
- The pretrained ResNet checkpoint and dataset require an internet connection on the first run.
- A GPU is recommended. Training on CPU may be slow.
- The input should be a clear image containing one handwritten digit. The model's image processor resizes it to the model's expected input size.
- Evaluation results are generated when the student runs the notebook; no fixed accuracy is guaranteed.